In [14]:
import os
import re
import pickle
import shutil
import unicodedata
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd

from rapidfuzz import fuzz

print("Notebook 11 initialized.")

Notebook 11 initialized.


In [15]:
# Cell 2 — locate project root

from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()

while not (
    (PROJECT_ROOT / "data" / "train").exists()
    and (PROJECT_ROOT / "output").exists()
):
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError(
            "Could not locate amazon-ml-challenge project root."
        )

    PROJECT_ROOT = PROJECT_ROOT.parent

TRAIN_DIR = PROJECT_ROOT / "data" / "train"
OUTPUT_DIR = PROJECT_ROOT / "output"

S1_PATH = TRAIN_DIR / "train_source1.tsv"
S2_PATH = TRAIN_DIR / "train_source2.tsv"
S3_PATH = TRAIN_DIR / "train_source3.tsv"
GT_PATH = TRAIN_DIR / "train_ground_truth.tsv"

BLOCKING_PATH = OUTPUT_DIR / "blocking_artifacts_v1.pkl"
VAL_IDS_PATH = OUTPUT_DIR / "validation_s1_ids.txt"

print("Project:", PROJECT_ROOT)

Project: /Users/bhavikesh/Documents/Hackathon/Amazon_ml/amazon-ml-challenge


In [16]:
with open(VAL_IDS_PATH, "r", encoding="utf-8") as f:
    validation_s1_ids = {
        line.strip()
        for line in f
        if line.strip()
    }

GT = pd.read_csv(
    GT_PATH,
    sep="\t",
    dtype=str,
    keep_default_na=False
)

GT_val = GT[
    GT["source1_entity_id"].isin(validation_s1_ids)
].copy()

true_targets_by_s1_id = {}

for row in GT_val.itertuples(index=False):
    if row.matched_entity_ids:
        true_targets_by_s1_id[row.source1_entity_id] = {
            x.strip()
            for x in row.matched_entity_ids.split(",")
            if x.strip()
        }
    else:
        true_targets_by_s1_id[row.source1_entity_id] = set()

print("Validation S1:", len(validation_s1_ids))
print("Validation GT rows:", len(GT_val))
print(
    "Validation true relationships:",
    sum(len(x) for x in true_targets_by_s1_id.values())
)



Validation S1: 220682
Validation GT rows: 220682
Validation true relationships: 764045


In [17]:
validation_s1_entity_ids = []

for chunk in pd.read_csv(
    S1_PATH,
    sep="\t",
    dtype=str,
    chunksize=100_000,
    keep_default_na=False
):
    for row in chunk.itertuples(index=False):
        if row.entity_id in validation_s1_ids:
            validation_s1_entity_ids.append(row.entity_id)

assert len(validation_s1_entity_ids) == len(validation_s1_ids)
assert len(set(validation_s1_entity_ids)) == len(validation_s1_entity_ids)

val_s1_id_to_idx = {
    entity_id: idx
    for idx, entity_id in enumerate(validation_s1_entity_ids)
}

val_s1_idx_to_id = {
    idx: entity_id
    for entity_id, idx in val_s1_id_to_idx.items()
}

print("Validation S1 index:", len(val_s1_id_to_idx))
print("First 5:", validation_s1_entity_ids[:5])

Validation S1 index: 220682
First 5: ['S1-785847572', 'S1-796227187', 'S1-982210261', 'S1-791867209', 'S1-72918323']


In [18]:
with open(BLOCKING_PATH, "rb") as f:
    blocking = pickle.load(f)

token_to_id = blocking["token_to_id"]

name_exact_map = blocking["name_exact_map"]

name_token_to_s1 = blocking["name_token_to_s1"]
address_token_to_s1 = blocking["address_token_to_s1"]

name_pair_to_s1 = blocking["name_pair_to_s1"]
address_pair_to_s1 = blocking["address_pair_to_s1"]

active_name_pairs = blocking["active_name_pairs"]
active_address_pairs = blocking["active_address_pairs"]

target_name_pair_frequency_2 = blocking[
    "target_name_pair_frequency_2"
]

target_address_pair_frequency_2 = blocking[
    "target_address_pair_frequency_2"
]

NAME_TOKEN_THRESHOLD = blocking[
    "NAME_TOKEN_FREQUENCY_THRESHOLD"
]

ADDRESS_TOKEN_THRESHOLD = blocking[
    "ADDRESS_TOKEN_FREQUENCY_THRESHOLD"
]

PAIR_S1_THRESHOLD = blocking[
    "PAIR_S1_THRESHOLD"
]

TARGET_PAIR_THRESHOLD = blocking[
    "TARGET_PAIR_THRESHOLD"
]
name_token_frequency = blocking[
    "name_token_frequency"
]

address_token_frequency = blocking[
    "address_token_frequency"
]

print("Blocker loaded.")
print(
    "Thresholds:",
    NAME_TOKEN_THRESHOLD,
    ADDRESS_TOKEN_THRESHOLD,
    PAIR_S1_THRESHOLD,
    TARGET_PAIR_THRESHOLD
)

Blocker loaded.
Thresholds: 10 10 2 500


In [19]:
MODEL_DIR = OUTPUT_DIR / "models_v1"

with open(
    MODEL_DIR / "logistic_model.pkl",
    "rb"
) as f:
    logistic_model = pickle.load(f)

with open(
    MODEL_DIR / "logistic_scaler.pkl",
    "rb"
) as f:
    logistic_scaler = pickle.load(f)

with open(
    MODEL_DIR / "xgboost_model.pkl",
    "rb"
) as f:
    xgb_model = pickle.load(f)

with open(
    MODEL_DIR / "lightgbm_model.pkl",
    "rb"
) as f:
    lgb_model = pickle.load(f)

with open(
    MODEL_DIR / "model_feature_names.pkl",
    "rb"
) as f:
    model_feature_names = pickle.load(f)

print("Models loaded.")
print("Feature count:", len(model_feature_names))
print(model_feature_names)

Models loaded.
Feature count: 16
['name_exact', 'name_ratio', 'name_token_ratio', 'name_token_jaccard', 'name_length_ratio', 'name_digit_overlap', 'address_exact', 'address_ratio', 'address_token_ratio', 'address_token_jaccard', 'address_length_ratio', 'address_digit_overlap', 'country_same', 'name_present_both', 'address_present_both', 'source_is_s3']


In [20]:
def normalize_text(value):
    value = "" if value is None else str(value)

    value = unicodedata.normalize(
        "NFKC",
        value
    )

    value = value.casefold()

    value = re.sub(
        r"[^\w\s]",
        " ",
        value,
        flags=re.UNICODE
    )

    value = re.sub(
        r"\s+",
        " ",
        value
    )

    return value.strip()


def get_tokens(value):
    normalized = normalize_text(value)

    if not normalized:
        return []

    return normalized.split()


def get_digit_tokens(value):
    if not value:
        return set()

    return set(
        re.findall(
            r"\d+",
            value
        )
    )


def length_ratio(a, b):
    if not a or not b:
        return 0.0

    return (
        min(len(a), len(b))
        /
        max(len(a), len(b))
    )


def token_jaccard(a, b):
    if not a and not b:
        return 0.0

    union = a | b

    if not union:
        return 0.0

    return len(a & b) / len(union)

In [21]:
N_VAL_S1 = len(validation_s1_entity_ids)

val_s1_name = np.empty(
    N_VAL_S1,
    dtype=object
)

val_s1_address = np.empty(
    N_VAL_S1,
    dtype=object
)

val_s1_country = np.empty(
    N_VAL_S1,
    dtype=object
)

val_s1_name_tokens = np.empty(
    N_VAL_S1,
    dtype=object
)

val_s1_address_tokens = np.empty(
    N_VAL_S1,
    dtype=object
)

val_s1_name_digits = np.empty(
    N_VAL_S1,
    dtype=object
)

val_s1_address_digits = np.empty(
    N_VAL_S1,
    dtype=object
)

filled = np.zeros(
    N_VAL_S1,
    dtype=bool
)

for chunk in pd.read_csv(
    S1_PATH,
    sep="\t",
    dtype=str,
    chunksize=100_000,
    keep_default_na=False
):
    for row in chunk.itertuples(index=False):

        idx = val_s1_id_to_idx.get(
            row.entity_id
        )

        if idx is None:
            continue

        name = normalize_text(
            row.business_name
        )

        address = normalize_text(
            row.business_address
        )

        country = row.country.casefold().strip()

        val_s1_name[idx] = name
        val_s1_address[idx] = address
        val_s1_country[idx] = country

        val_s1_name_tokens[idx] = set(
            get_tokens(name)
        )

        val_s1_address_tokens[idx] = set(
            get_tokens(address)
        )

        val_s1_name_digits[idx] = (
            get_digit_tokens(name)
        )

        val_s1_address_digits[idx] = (
            get_digit_tokens(address)
        )

        filled[idx] = True

assert filled.all()

print(
    "Validation S1 records loaded:",
    int(filled.sum())
)

Validation S1 records loaded: 220682


In [22]:
def generate_validation_candidates(
    target_name,
    target_address
):
    candidate_s1 = set()

    # --------------------------------------------------------
    # Normalize
    # --------------------------------------------------------

    normalized_name = normalize_text(
        target_name
    )

    normalized_address = normalize_text(
        target_address
    )

    # --------------------------------------------------------
    # Token IDs
    # --------------------------------------------------------

    name_tokens = set(
        get_tokens(normalized_name)
    )

    address_tokens = set(
        get_tokens(normalized_address)
    )

    name_token_ids = sorted(
        token_to_id[token]
        for token in name_tokens
        if token in token_to_id
    )

    address_token_ids = sorted(
        token_to_id[token]
        for token in address_tokens
        if token in token_to_id
    )

    # --------------------------------------------------------
    # 1. Exact normalized name
    # --------------------------------------------------------

    if normalized_name:
        candidate_s1.update(
            name_exact_map.get(
                normalized_name,
                []
            )
        )

    # --------------------------------------------------------
    # 2. Name-token blocking
    # IMPORTANT: only use tokens with frequency <= threshold
    # --------------------------------------------------------

    for token in name_tokens:

        freq = name_token_frequency.get(
            token,
            0
        )

        if 0 < freq <= NAME_TOKEN_THRESHOLD:
            candidate_s1.update(
                name_token_to_s1.get(
                    token,
                    []
                )
            )

    # --------------------------------------------------------
    # 3. Address-token blocking
    # IMPORTANT: only use tokens with frequency <= threshold
    # --------------------------------------------------------

    for token in address_tokens:

        freq = address_token_frequency.get(
            token,
            0
        )

        if 0 < freq <= ADDRESS_TOKEN_THRESHOLD:
            candidate_s1.update(
                address_token_to_s1.get(
                    token,
                    []
                )
            )

    # --------------------------------------------------------
    # 4. S1-side name pair blocking
    # --------------------------------------------------------

    if len(name_token_ids) >= 2:

        for pair in combinations(
            name_token_ids,
            2
        ):

            if pair in active_name_pairs:
                candidate_s1.update(
                    name_pair_to_s1.get(
                        pair,
                        []
                    )
                )

    # --------------------------------------------------------
    # 5. S1-side address pair blocking
    # --------------------------------------------------------

    if len(address_token_ids) >= 2:

        for pair in combinations(
            address_token_ids,
            2
        ):

            if pair in active_address_pairs:
                candidate_s1.update(
                    address_pair_to_s1.get(
                        pair,
                        []
                    )
                )

    # --------------------------------------------------------
    # 6. Target-side name pair <= TARGET_PAIR_THRESHOLD
    # --------------------------------------------------------

    if len(name_token_ids) >= 2:

        for pair in combinations(
            name_token_ids,
            2
        ):

            freq = target_name_pair_frequency_2.get(
                pair
            )

            if (
                freq is not None
                and 0 < freq <= TARGET_PAIR_THRESHOLD
            ):
                candidate_s1.update(
                    name_pair_to_s1.get(
                        pair,
                        []
                    )
                )

    # --------------------------------------------------------
    # 7. Target-side address pair <= TARGET_PAIR_THRESHOLD
    # --------------------------------------------------------

    if len(address_token_ids) >= 2:

        for pair in combinations(
            address_token_ids,
            2
        ):

            freq = target_address_pair_frequency_2.get(
                pair
            )

            if (
                freq is not None
                and 0 < freq <= TARGET_PAIR_THRESHOLD
            ):
                candidate_s1.update(
                    address_pair_to_s1.get(
                        pair,
                        []
                    )
                )

    return sorted(candidate_s1)

In [23]:
def build_pair_features(
    candidate_indices,
    target_name,
    target_address,
    target_country,
    source_number
):
    target_name = normalize_text(
        target_name
    )

    target_address = normalize_text(
        target_address
    )

    target_country = (
        target_country.casefold().strip()
    )

    target_name_tokens = set(
        get_tokens(target_name)
    )

    target_address_tokens = set(
        get_tokens(target_address)
    )

    target_name_digits = (
        get_digit_tokens(target_name)
    )

    target_address_digits = (
        get_digit_tokens(target_address)
    )

    n = len(candidate_indices)

    X = np.empty(
        (n, 16),
        dtype=np.float32
    )

    for j, s1_idx_value in enumerate(
        candidate_indices
    ):

        s1_n = val_s1_name[s1_idx_value]
        s1_a = val_s1_address[s1_idx_value]
        s1_c = val_s1_country[s1_idx_value]

        s1_nt = val_s1_name_tokens[
            s1_idx_value
        ]

        s1_at = val_s1_address_tokens[
            s1_idx_value
        ]

        s1_nd = val_s1_name_digits[
            s1_idx_value
        ]

        s1_ad = val_s1_address_digits[
            s1_idx_value
        ]

        name_exact = int(
            bool(s1_n)
            and s1_n == target_name
        )

        name_ratio = (
            fuzz.ratio(
                s1_n,
                target_name
            ) / 100.0
            if s1_n and target_name
            else 0.0
        )

        name_token_ratio = (
            fuzz.token_set_ratio(
                s1_n,
                target_name
            ) / 100.0
            if s1_n and target_name
            else 0.0
        )

        name_jaccard = token_jaccard(
            s1_nt,
            target_name_tokens
        )

        name_len_ratio = length_ratio(
            s1_n,
            target_name
        )

        name_digit_overlap = len(
            s1_nd
            &
            target_name_digits
        )

        address_exact = int(
            bool(s1_a)
            and s1_a == target_address
        )

        address_ratio = (
            fuzz.ratio(
                s1_a,
                target_address
            ) / 100.0
            if s1_a and target_address
            else 0.0
        )

        address_token_ratio = (
            fuzz.token_set_ratio(
                s1_a,
                target_address
            ) / 100.0
            if s1_a and target_address
            else 0.0
        )

        address_jaccard = token_jaccard(
            s1_at,
            target_address_tokens
        )

        address_len_ratio = length_ratio(
            s1_a,
            target_address
        )

        address_digit_overlap = len(
            s1_ad
            &
            target_address_digits
        )

        country_same = int(
            bool(s1_c)
            and s1_c == target_country
        )

        name_present_both = int(
            bool(s1_n and target_name)
        )

        address_present_both = int(
            bool(s1_a and target_address)
        )

        X[j] = [
            name_exact,
            name_ratio,
            name_token_ratio,
            name_jaccard,
            name_len_ratio,
            name_digit_overlap,

            address_exact,
            address_ratio,
            address_token_ratio,
            address_jaccard,
            address_len_ratio,
            address_digit_overlap,

            country_same,
            name_present_both,
            address_present_both,

            1.0 if source_number == 3 else 0.0
        ]

    return X

In [24]:
# Cell 11 — Full validation inference and threshold evaluation

import numpy as np
import pandas as pd
from pathlib import Path

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

THRESHOLDS = np.round(
    np.arange(0.680, 0.7801, 0.0025),
    4
)

N_THRESHOLDS = len(THRESHOLDS)
N_BINS = N_THRESHOLDS + 1

BLOCK_PAIRS = 500_000

print("Full validation inference")
print("Validation S1:", N_VAL_S1)
print("Thresholds:", N_THRESHOLDS)
print("Range:", THRESHOLDS[0], "to", THRESHOLDS[-1])

# ------------------------------------------------------------
# Ground-truth counts per validation S1
# ------------------------------------------------------------

true_counts = np.fromiter(
    (
        len(
            true_targets_by_s1_id[s1_id]
        )
        for s1_id in validation_s1_entity_ids
    ),
    dtype=np.int32,
    count=N_VAL_S1
)

# ------------------------------------------------------------
# Reverse target -> true validation S1 index
# ------------------------------------------------------------
# A target can belong to at most one validation S1 entity.
# This was verified during the earlier GT analysis.

true_target_to_s1_idx = {}

for s1_id, target_ids in true_targets_by_s1_id.items():

    s1_idx_value = val_s1_id_to_idx[s1_id]

    for target_id in target_ids:

        previous = true_target_to_s1_idx.get(
            target_id
        )

        if (
            previous is not None
            and previous != s1_idx_value
        ):
            raise ValueError(
                f"Target {target_id} maps to "
                f"multiple validation S1 entities."
            )

        true_target_to_s1_idx[target_id] = (
            s1_idx_value
        )

print(
    "Validation true target IDs:",
    len(true_target_to_s1_idx)
)

# ------------------------------------------------------------
# Histograms
#
# For every S1 we store how many candidate scores fall into
# each threshold bucket.
#
# Bucket 0:
#   score < threshold[0]
#
# Bucket k:
#   threshold[k-1] <= score < threshold[k]
#
# Final bucket:
#   score >= threshold[-1]
# ------------------------------------------------------------

prediction_hist = np.zeros(
    (N_VAL_S1, N_BINS),
    dtype=np.int32
)

true_positive_hist = np.zeros(
    (N_VAL_S1, N_BINS),
    dtype=np.int32
)

# ------------------------------------------------------------
# Temporary block storage
# ------------------------------------------------------------

block_s1 = []
block_scores = []
block_truth = []

block_size = 0

total_candidates = 0
total_positive_candidates = 0

processed_rows = {
    "S2": 0,
    "S3": 0
}

# ------------------------------------------------------------
# Histogram update helper
# ------------------------------------------------------------

def flush_score_block():

    global block_s1
    global block_scores
    global block_truth
    global block_size
    global prediction_hist
    global true_positive_hist

    if block_size == 0:
        return

    s1_values = np.concatenate(
        block_s1
    ).astype(
        np.int64,
        copy=False
    )

    score_values = np.concatenate(
        block_scores
    ).astype(
        np.float32,
        copy=False
    )

    truth_values = np.concatenate(
        block_truth
    ).astype(
        np.uint8,
        copy=False
    )

    # Number of thresholds each score passes.
    threshold_bins = np.searchsorted(
        THRESHOLDS,
        score_values,
        side="right"
    ).astype(
        np.int64,
        copy=False
    )

    flat_indices = (
        s1_values * N_BINS
        + threshold_bins
    )

    block_prediction_hist = np.bincount(
        flat_indices,
        minlength=N_VAL_S1 * N_BINS
    ).reshape(
        N_VAL_S1,
        N_BINS
    )

    prediction_hist += (
        block_prediction_hist.astype(
            np.int32,
            copy=False
        )
    )

    positive_mask = (
        truth_values == 1
    )

    if positive_mask.any():

        positive_flat_indices = flat_indices[
            positive_mask
        ]

        block_true_positive_hist = np.bincount(
            positive_flat_indices,
            minlength=N_VAL_S1 * N_BINS
        ).reshape(
            N_VAL_S1,
            N_BINS
        )

        true_positive_hist += (
            block_true_positive_hist.astype(
                np.int32,
                copy=False
            )
        )

    block_s1.clear()
    block_scores.clear()
    block_truth.clear()
    block_size = 0


# ------------------------------------------------------------
# Full S2 + S3 scan
# ------------------------------------------------------------

for source_number, source_path, source_name in [
    (2, S2_PATH, "S2"),
    (3, S3_PATH, "S3"),
]:

    source_candidates = 0

    for chunk_number, chunk in enumerate(
        pd.read_csv(
            source_path,
            sep="\t",
            dtype=str,
            chunksize=100_000,
            keep_default_na=False
        ),
        start=1
    ):

        for row in chunk.itertuples(index=False):

            processed_rows[source_name] += 1

            candidates = generate_validation_candidates(
                row.business_name,
                row.business_address
            )

            if not candidates:
                continue

            X_pair = build_pair_features(
                candidates,
                row.business_name,
                row.business_address,
                row.country,
                source_number
            )

            # Logistic is the selected model from the pilot.
            logistic_scores = logistic_model.predict_proba(
                logistic_scaler.transform(X_pair)
            )[:, 1]

            candidate_array = np.asarray(
                candidates,
                dtype=np.int64
            )

            true_s1_idx_value = (
                true_target_to_s1_idx.get(
                    row.entity_id
                )
            )

            if true_s1_idx_value is None:

                truth = np.zeros(
                    len(candidate_array),
                    dtype=np.uint8
                )

            else:

                truth = (
                    candidate_array
                    == true_s1_idx_value
                ).astype(
                    np.uint8
                )

            block_s1.append(
                candidate_array
            )

            block_scores.append(
                logistic_scores.astype(
                    np.float32,
                    copy=False
                )
            )

            block_truth.append(
                truth
            )

            current_count = len(
                candidate_array
            )

            block_size += current_count
            source_candidates += current_count
            total_candidates += current_count
            total_positive_candidates += int(
                truth.sum()
            )

            if block_size >= BLOCK_PAIRS:
                flush_score_block()

        print(
            f"{source_name}: "
            f"{processed_rows[source_name]:,} rows | "
            f"{source_candidates:,} candidates"
        )

    # Flush any remaining pairs for this source.
    flush_score_block()

    print(
        f"{source_name} COMPLETE | "
        f"rows={processed_rows[source_name]:,} | "
        f"candidates={source_candidates:,}"
    )

# ------------------------------------------------------------
# Convert histogram buckets to cumulative prediction counts
# ------------------------------------------------------------
#
# For threshold t:
#   predicted = score >= t
#
# ------------------------------------------------------------

prediction_cumulative = np.cumsum(
    prediction_hist[:, ::-1],
    axis=1
)[:, ::-1]

true_positive_cumulative = np.cumsum(
    true_positive_hist[:, ::-1],
    axis=1
)[:, ::-1]

# Ignore bucket 0 because it represents:
# score < smallest threshold.
#
# Columns 1..N_THRESHOLDS correspond to our thresholds.

predicted_counts = prediction_cumulative[
    :,
    1:
]

true_positive_counts = true_positive_cumulative[
    :,
    1:
]

# ------------------------------------------------------------
# Macro F0.5 for every threshold
# ------------------------------------------------------------

results = []

for threshold_index, threshold in enumerate(
    THRESHOLDS
):

    predicted = predicted_counts[
        :,
        threshold_index
    ].astype(
        np.float64
    )

    true_positive = true_positive_counts[
        :,
        threshold_index
    ].astype(
        np.float64
    )

    f_scores = np.zeros(
        N_VAL_S1,
        dtype=np.float64
    )

    # --------------------------------------------------------
    # Correct singleton with empty prediction
    # --------------------------------------------------------

    correct_singleton = (
        (true_counts == 0)
        &
        (predicted == 0)
    )

    f_scores[
        correct_singleton
    ] = 1.0

    # --------------------------------------------------------
    # Entities with both true matches and predictions
    # --------------------------------------------------------

    active = (
        (true_counts > 0)
        &
        (predicted > 0)
    )

    precision = np.zeros(
        N_VAL_S1,
        dtype=np.float64
    )

    recall = np.zeros(
        N_VAL_S1,
        dtype=np.float64
    )

    precision[active] = (
        true_positive[active]
        / predicted[active]
    )

    recall[active] = (
        true_positive[active]
        / true_counts[active]
    )

    denominator = (
        0.25 * precision
        + recall
    )

    valid = (
        active
        &
        (denominator > 0)
    )

    f_scores[valid] = (
        1.25
        * precision[valid]
        * recall[valid]
        / denominator[valid]
    )

    # --------------------------------------------------------
    # Macro average across ALL validation S1 entities
    # --------------------------------------------------------

    macro_f05 = f_scores.mean()

    nonempty_prediction_rate = (
        100.0
        * (predicted > 0).mean()
    )

    mean_predictions = (
        predicted.mean()
    )

    results.append(
        {
            "threshold": threshold,
            "macro_f05": macro_f05,
            "nonempty_prediction_pct":
                nonempty_prediction_rate,
            "mean_predictions_per_s1":
                mean_predictions,
            "singleton_count":
                int((true_counts == 0).sum()),
            "singleton_correct_empty":
                int(
                    correct_singleton.sum()
                )
        }
    )

validation_results = pd.DataFrame(
    results
)

best_row = validation_results.loc[
    validation_results["macro_f05"].idxmax()
]

# ------------------------------------------------------------
# Save reusable validation artifacts
# ------------------------------------------------------------

VALIDATION_DIR = (
    OUTPUT_DIR
    / "validation_logistic_v1"
)

VALIDATION_DIR.mkdir(
    parents=True,
    exist_ok=True
)

np.save(
    VALIDATION_DIR
    / "prediction_hist.npy",
    prediction_hist
)

np.save(
    VALIDATION_DIR
    / "true_positive_hist.npy",
    true_positive_hist
)

validation_results.to_csv(
    VALIDATION_DIR
    / "threshold_results.csv",
    index=False
)

with open(
    VALIDATION_DIR
    / "best_threshold.txt",
    "w",
    encoding="utf-8"
) as f:
    f.write(
        str(best_row["threshold"])
    )

# ------------------------------------------------------------
# Final report
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FULL VALIDATION COMPLETE")
print("=" * 70)

print(
    "S2 rows:",
    f"{processed_rows['S2']:,}"
)

print(
    "S3 rows:",
    f"{processed_rows['S3']:,}"
)

print(
    "Total candidate pairs:",
    f"{total_candidates:,}"
)

print(
    "Candidate true positives:",
    f"{total_positive_candidates:,}"
)

print(
    "\nBest Logistic threshold:",
    best_row["threshold"]
)

print(
    "Best macro F0.5:",
    best_row["macro_f05"]
)

print(
    "Nonempty prediction rate:",
    f"{best_row['nonempty_prediction_pct']:.2f}%"
)

print(
    "Mean predictions per S1:",
    best_row["mean_predictions_per_s1"]
)

print("\nTop 10 thresholds:")

display(
    validation_results.sort_values(
        "macro_f05",
        ascending=False
    ).head(10)
)

print(
    "\nSaved validation artifacts to:",
    VALIDATION_DIR
)

Full validation inference
Validation S1: 220682
Thresholds: 41
Range: 0.68 to 0.78
Validation true target IDs: 764045
S2: 100,000 rows | 565,051 candidates
S2: 200,000 rows | 1,132,365 candidates
S2: 300,000 rows | 1,701,644 candidates
S2: 400,000 rows | 2,267,112 candidates
S2: 500,000 rows | 2,832,892 candidates
S2: 600,000 rows | 3,398,627 candidates
S2: 700,000 rows | 3,962,795 candidates
S2: 800,000 rows | 4,526,468 candidates
S2: 900,000 rows | 5,095,207 candidates
S2: 1,000,000 rows | 5,662,638 candidates
S2: 1,100,000 rows | 6,231,317 candidates
S2: 1,200,000 rows | 6,798,250 candidates
S2: 1,300,000 rows | 7,362,626 candidates
S2: 1,400,000 rows | 7,926,010 candidates
S2: 1,500,000 rows | 8,494,160 candidates
S2: 1,600,000 rows | 9,059,225 candidates
S2: 1,700,000 rows | 9,623,072 candidates
S2: 1,800,000 rows | 10,186,481 candidates
S2: 1,900,000 rows | 10,748,919 candidates
S2: 2,000,000 rows | 11,314,789 candidates
S2: 2,100,000 rows | 11,882,266 candidates
S2: 2,200,000 ro

,threshold,macro_f05,nonempty_prediction_pct,mean_predictions_per_s1,singleton_count,singleton_correct_empty
15,0.7175,0.217143,31.673630,0.639277,12453,11182
16,0.7200,0.216979,29.816206,0.542233,12453,11471
14,0.7150,0.215746,33.968335,0.794609,12453,10809
17,0.7225,0.215604,28.363890,0.480787,12453,11676
18,0.7250,0.213775,27.226507,0.441925,12453,11810
13,0.7125,0.212491,36.639146,1.037833,12453,10388
19,0.7275,0.211478,26.356930,0.416273,12453,11886
20,0.7300,0.209461,25.708486,0.399117,12453,11944
21,0.7325,0.207329,25.158373,0.386905,12453,11989
12,0.7100,0.206746,39.716878,1.409281,12453,9831



Saved validation artifacts to: /Users/bhavikesh/Documents/Hackathon/Amazon_ml/amazon-ml-challenge/output/validation_logistic_v1
